# Эффект шаттла: доступность порта и станции «до» и «после»

Методика: пешеходный граф УДС (5 км/ч) из `scripts/walk_graph.py`; «после» — маршрут шаттла рёбрами на
25 км/ч с пешеходными подходами к 4 остановкам. Допущение: без ожидания и расписания (верхняя оценка
эффекта). Цель верификации — старая колода: «90→40 мин до порта, 0→1 171 жит. в 30 мин от порта,
32→26 мин до станции, 78→88% города».

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from shapely.geometry import Point

from walk_graph import (CRS, MCRS, SPEED_WALK, ROOT, build_walk_graph, graph_with_shuttle,
                        nearest_node)

DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "data" / "results"
blocks = gpd.read_file(DATA / "blocks.geojson")
buildings = gpd.read_file(DATA / "building.geojson")

G, pos_m, names = build_walk_graph()
print(f"граф: {G.number_of_nodes()} узлов, {G.number_of_edges()} рёбер")

граф: 4366 узлов, 4453 рёбер


In [2]:
# --- население по кварталам (как в тетради 01)
cent = buildings.to_crs(MCRS).geometry.centroid
pop_pts = gpd.GeoDataFrame({"population": buildings["population"].values}, geometry=cent, crs=MCRS).to_crs(CRS)
bj = gpd.sjoin(blocks[["geometry"]], pop_pts, predicate="contains")
pop_blk = bj.groupby(level=0)["population"].sum().reindex(blocks.index, fill_value=0)
blk_m = blocks.to_crs(MCRS).geometry.centroid
print("население в кварталах:", int(pop_blk.sum()))

население в кварталах: 13597


In [3]:
# --- время до цели: «до» — пешком; «после» — пешком + шаттл (без ожидания)
G2, pos_m2, stop_nodes = graph_with_shuttle(G, pos_m)
print(f"граф 'после': +{G2.number_of_nodes() - G.number_of_nodes()} узлов (остановки), "
      f"+{G2.number_of_edges() - G.number_of_edges()} рёбер (шаттл и подходы)")

port_m = gpd.GeoSeries([Point(28.7355, 60.3500)], crs=CRS).to_crs(MCRS).iloc[0]
station_m = gpd.GeoSeries([Point(28.6309, 60.3634)], crs=CRS).to_crs(MCRS).iloc[0]

src_nodes = [nearest_node(pos_m, p) for p in blk_m]
src_after = [nearest_node(pos_m2, p) for p in blk_m]

def times_to(target_m, graph, pos, srcs):
    src = nearest_node(pos, target_m)
    d = nx.single_source_dijkstra_path_length(graph, src, weight="weight")
    return np.array([d.get(s, np.inf) + abs(pos[s].distance(p)) / SPEED_WALK
                     for s, p in zip(srcs, blk_m)])

t_port_before = times_to(port_m, G, pos_m, src_nodes)
t_port_after = times_to(port_m, G2, pos_m2, src_after)
t_stat_before = times_to(station_m, G, pos_m, src_nodes)
t_stat_after = times_to(station_m, G2, pos_m2, src_after)
print("посчитано")

граф 'после': +4 узлов (остановки), +7 рёбер (шаттл и подходы)
посчитано


In [4]:
# --- метрики (взвешенно по населению квартала)
w = pop_blk.values.astype(float)
def wmedian(t):
    ok = np.isfinite(t) & (w > 0)
    order = np.argsort(t[ok])
    c = np.cumsum(w[ok][order]) / w[ok].sum()
    return float(t[ok][order][np.searchsorted(c, 0.5)])

def share30(t):
    ok = np.isfinite(t)
    return float(w[ok & (t <= 30)].sum() / w[ok].sum())

rows = []
for nm, tb, ta in (("до порта", t_port_before, t_port_after), ("до станции", t_stat_before, t_stat_after)):
    rows.append({"цель": nm,
                 "медиана_до_мин": round(wmedian(tb)), "медиана_после_мин": round(wmedian(ta)),
                 "среднее_до_мин": round(float(np.average(tb[~np.isinf(tb)], weights=w[~np.isinf(tb)]))),
                 "среднее_после_мин": round(float(np.average(ta[~np.isinf(ta)], weights=w[~np.isinf(ta)]))),
                 "доля_30мин_до": round(share30(tb), 2), "доля_30мин_после": round(share30(ta), 2),
                 "жит_30мин_до": int(w[~np.isinf(tb) & (tb <= 30)].sum()),
                 "жит_30мин_после": int(w[~np.isinf(ta) & (ta <= 30)].sum())})
effect = pd.DataFrame(rows)
effect.to_csv(RESULTS / "shuttle_effect.csv", index=False, encoding="utf-8-sig")
effect

,цель,медиана_до_мин,медиана_после_мин,среднее_до_мин,среднее_после_мин,доля_30мин_до,доля_30мин_после,жит_30мин_до,жит_30мин_после
0,до порта,184,167,209,181,0.02,0.17,168,1856
1,до станции,167,162,204,177,0.25,0.31,2731,3319


In [6]:
# --- наглядно: от центра Приморска (остановка «Шаттл: центр») до порта и станции
from walk_graph import SPEED_WALK
center_m = gpd.GeoSeries([Point(28.6085, 60.3649)], crs=CRS).to_crs(MCRS).iloc[0]
station_m = gpd.GeoSeries([Point(28.6309, 60.3634)], crs=CRS).to_crs(MCRS).iloc[0]
port_m = gpd.GeoSeries([Point(28.7355, 60.3500)], crs=CRS).to_crs(MCRS).iloc[0]

def point_time(target_m, src_m, graph, pos):
    d = nx.single_source_dijkstra_path_length(graph, nearest_node(pos, target_m), weight="weight")
    s = nearest_node(pos, src_m)
    return d.get(s, np.inf) + pos[s].distance(src_m) / SPEED_WALK

print(f"центр → порт: {point_time(port_m, center_m, G, pos_m):.0f} мин пешком → "
      f"{point_time(port_m, center_m, G2, pos_m2):.0f} мин с шаттлом")
print(f"центр → станция: {point_time(station_m, center_m, G, pos_m):.0f} → "
      f"{point_time(station_m, center_m, G2, pos_m2):.0f} мин")
print("старая колода (для сверки): порт 90→40 мин, станция 32→26 мин")

центр → порт: 101 мин пешком → 23 мин с шаттлом
центр → станция: 22 → 4 мин
старая колода (для сверки): порт 90→40 мин, станция 32→26 мин
